# Reading t-digest products: tensors and raw values

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/tdigest_reader_example.ipynb)

_Runs end-to-end on [Binder](https://mybinder.org/v2/gh/englacial/zagg/main?urlpath=lab/tree/notebooks/tdigest_reader_example.ipynb): it builds a small synthetic t-digest store in memory -- no cloud data, no credentials. The Binder image already provides `zagg[analysis]` (incl. matplotlib) via the repo's `.binder/` environment._

zagg writes a gridded Zarr product where each coverage chunk is a `64 x 64`
block of child cells, and each populated cell carries a **t-digest** sketch of
its observed values (e.g. ICESat-2 photon elevations). The sketch is stored in a
ragged **variable-length-bytes** field (issue #209): one vlen array on the cell
grid whose populated cells each hold the raw little-endian bytes of their
`(k_centroids, 2)` `(mean, weight)` digest. The layout is self-describing -- see
the [ragged layout doc](https://github.com/englacial/zagg/blob/main/docs/ragged_layout.md).

A downstream client usually wants something denser and fixed-size. The
`zagg.readers` package (issue #79) reconstructs that from the stored digests:

- **`read_tensors`** -- a generator yielding `(tensor, morton_index)` per chunk,
  where `tensor` has shape `(64, 64, n_bins)`: each cell's digest rasterized into
  `n_bins` evenly-spaced z-bins.
- **`read_raw_values`** -- the lossless companion: when a cell's digest was built
  with no merges, its centroid means *are* the original samples, so the raw value
  vector is recovered exactly (and it raises when that is not possible).
- **`read_cell`** -- random access to one cell's digest (2 ranged GETs on a
  sharded product, never the whole shard).

This notebook is **self-contained**: it builds a small synthetic t-digest store
in memory, so it runs anywhere (including Binder) with no external data or
credentials. The store is built with the same public API zagg uses to write the
real product (`zagg.stats.tdigest.build_tdigest` +
`zagg.processing.write_ragged_to_zarr`), so the read path exercised here is
identical to the one a real product uses.

## 0. Requirements

The reader API (`read_tensors` / `read_raw_values`) needs only zagg's core
dependencies (numpy + zarr). The one plot below uses `matplotlib`, which ships
in zagg's `analysis` extra.

- **On Binder** nothing to do -- the repo's `.binder/` environment already
  installs `zagg[analysis]` (matplotlib included), so every cell runs as-is.
- **Standalone** (your own laptop/hub), install once with
  `pip install 'zagg[analysis]'` before running the notebook.

## 1. Build a synthetic t-digest store

We mimic two coverage chunks at real order-6 morton ids, each with a handful of
populated cells. For every populated cell we draw some samples and sketch them
with `build_tdigest`, then write the chunk's per-cell digests into the product's
ragged vlen array with `write_ragged_to_zarr` -- exactly the layout the readers
consume. Each chunk's coverage-cell morton id is recovered by the readers from
the sibling `morton` coordinate array (not from a subgroup name).

A real product would instead point the readers at an on-disk or S3-backed Zarr
store of the same shape; nothing else about the read code changes.

In [ ]:
import numpy as np
import zarr
from zarr.storage import MemoryStore

from zagg.config import PipelineConfig
from zagg.grids import HealpixGrid
from zagg.grids.morton import morton_word
from zagg.processing import write_ragged_to_zarr
from zagg.stats.tdigest import build_tdigest

# One ragged t-digest field on a HEALPix grid: order-6 shards (parent), order-12
# cells (child) -> each 4096-cell coverage chunk is a 64x64 child block.
config = PipelineConfig(
    data_source={"groups": ["g"]},
    aggregation={
        "coordinates": {"morton": {"dtype": "uint64", "fill_value": 0}},
        "variables": {
            "h_tdigest": {
                "function": "zagg.stats.tdigest.build_tdigest",
                "source": "h",
                "kind": "ragged",
                "inner_shape": [2],  # each centroid is a (mean, weight) pair
                "dtype": "float32",
                "fill_value": 0,
            }
        },
    },
    output={"grid": {"type": "healpix", "parent_order": 6, "child_order": 12}},
)
grid = HealpixGrid(6, 12, layout="fullsphere", config=config)
FIELD = f"{grid.group_path}/h_tdigest"  # array path the readers consume, e.g. "12/h_tdigest"
rng = np.random.default_rng(0)


def write_chunk(store, morton_key, cell_to_values, *, delta=512):
    """Sketch each cell's samples and write the chunk into the ragged vlen array.

    Returns ``(morton_word, cell_base)`` -- the chunk's packed coverage-cell id
    and the global offset of its first cell (so single cells can be addressed by
    ``read_cell`` below).
    """
    word = morton_word(morton_key)
    block = grid.block_index(word)
    base = block[0] * grid.cells_per_chunk
    # The dense per-cell morton coordinate the readers use for chunk identity.
    morton_arr = zarr.open_array(store, path=f"{grid.group_path}/morton", mode="r+")
    morton_arr[base : base + grid.cells_per_chunk] = grid.children(word)
    cell_ids = sorted(cell_to_values)
    digests = [build_tdigest(np.asarray(cell_to_values[c]), delta=delta) for c in cell_ids]
    write_ragged_to_zarr({"h_tdigest": (digests, cell_ids)}, store, grid=grid, chunk_idx=block)
    return word, base


store = MemoryStore()
grid.emit_template(store)

# Chunk A: elevations clustered around ~20 m. cell_id is the cell's position in
# the chunk's row-major (64x64) children block (so 0 -> (0,0), 4095 -> (63,63)).
word_a, base_a = write_chunk(
    store,
    "1121121",
    {
        0: rng.normal(20.0, 2.0, 3_000),
        5: rng.normal(22.0, 1.5, 2_000),
        4095: rng.normal(19.0, 2.5, 1_500),
    },
)

# Chunk B: a different elevation band, ~50 m.
word_b, base_b = write_chunk(
    store,
    "2431123",
    {
        7: rng.normal(50.0, 3.0, 2_500),
        63: rng.normal(52.0, 2.0, 2_000),
    },
)

print("chunk morton ids written:", sorted((word_a, word_b)))

The field is now ONE `variable_length_bytes` array on the cell grid: each
populated cell holds the raw little-endian bytes of its `(k_centroids, 2)`
`(mean, weight)` digest, and empty cells keep the `b""` fill. The element
interpretation (dtype + shape) rides in the array's `ragged` attrs, so a reader
decodes exactly what the writer declared. We can peek at one cell's digest with
`read_cell`, which random-accesses a single cell.

In [ ]:
from zagg.readers import read_cell

# read_cell addresses ONE cell by its global position (chunk base + cell id).
digest = read_cell(store, FIELD, base_a + 5)  # cell 5 of chunk A
print(f"cell 5 of chunk A: digest shape {digest.shape} (k_centroids, [mean, weight]):")
print(digest[:5])

## 2. `read_tensors`: dense fixed-size tensors

`read_tensors` yields `(tensor, morton_index)` per chunk. For each chunk it:

1. trims each cell's tails to the `bottom`/`top` quantiles (default 5%/95%),
2. anchors a fixed `n_bins * resolution` z-window at the floor of the trimmed
   range,
3. rasterizes every cell's digest into `n_bins` per-bin counts over that window,
4. emits the `(64, 64, n_bins)` tensor plus the chunk's morton id (recovered
   from the store).

Defaults: `n_bins=128`, `resolution=0.5` m (a 64 m window), `dtype="uint32"`.

In [ ]:
from zagg.readers import read_tensors

tensors = {morton: tensor for tensor, morton in read_tensors(store, FIELD)}

for morton, tensor in sorted(tensors.items()):
    nonzero_cells = int((tensor.sum(axis=2) > 0).sum())
    print(
        f"chunk morton={morton}: tensor {tensor.shape} dtype={tensor.dtype}, "
        f"{nonzero_cells} populated cells, total counts={int(tensor.sum())}"
    )

The morton id is the chunk's coverage cell, recovered from the sibling
`{field}/morton` coordinate array (the per-cell morton word coarsened to the
chunk's order). Populated cells land at their row-major position; everything
else is zero.

In [ ]:
tensor = tensors[word_a]
# cell 5 -> (row 0, col 5); cell 4095 -> (row 63, col 63); cell 10 unpopulated.
print("cell 5   (0, 5)  count sum:", int(tensor[0, 5].sum()))
print("cell 4095 (63,63) count sum:", int(tensor[63, 63].sum()))
print("cell 10  (0,10) count sum:", int(tensor[0, 10].sum()), "(unpopulated)")

### The reconstructed histogram

Each cell's length-`n_bins` vector is a histogram reconstructed from the digest
CDF. Plotting one cell's vector against the z-bin edges shows the recovered
distribution -- here a roughly Gaussian elevation profile centred near 20 m.

In [ ]:
import matplotlib.pyplot as plt

from zagg.readers import chunk_z_range

n_bins, resolution = 128, 0.5
tensor = tensors[word_a]
vec = tensor[0, 5]  # the digest histogram for cell 5 of chunk A

# Reconstruct the window the reader used: floor of the chunk's trimmed minimum.
# We recompute the floor here only to label the x-axis; the reader sets it internally.
digests = [read_cell(store, FIELD, base_a + c) for c in (0, 5, 4095)]  # chunk A's cells
z_lo, n_bins_c, res_c = chunk_z_range(
    digests, n_bins=n_bins, resolution=resolution, bottom=0.05, top=0.95, fit="raise"
)
edges = z_lo + res_c * np.arange(n_bins_c + 1)
centers = 0.5 * (edges[:-1] + edges[1:])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(centers, vec, width=res_c, align="center", color="steelblue", edgecolor="none")
ax.set_xlabel("z (m)")
ax.set_ylabel("count")
ax.set_title(f"cell 5 of chunk A -- reconstructed histogram (z_lo={z_lo:.0f} m)")
ax.set_xlim(z_lo, z_lo + 30)  # zoom to the populated region
plt.tight_layout()
plt.show()

### Output dtype flag

The client consumes `uint16`; `read_tensors` also offers `uint32` (the default,
safe for dense cells with many counts per bin) and `float32` (keeps fractional
reconstructed counts). The flag only changes the output cast.

In [ ]:
for dtype in ("uint16", "uint32", "float32"):
    t, _ = next(read_tensors(store, FIELD, dtype=dtype))
    print(f"dtype={dtype:8s} -> tensor.dtype={t.dtype}, peak bin count={t.max()}")

## 3. Fit policy: when the trimmed range exceeds the window

The window is fixed at `n_bins * resolution` (64 m by default). If a chunk's
trimmed `bottom`->`top` range is wider than that, the `fit` flag decides:

- `"raise"` (default) -- raise, so the client never silently loses tails;
- `"degrade_resolution"` -- double `resolution` in powers of two until it fits
  (keeps `n_bins`);
- `"collapse_bins"` -- shrink `n_bins` to the smallest power of two that fits
  (keeps `resolution`).

We build a deliberately wide chunk (elevations spanning ~400 m) to trigger it.

In [ ]:
wide = MemoryStore()
grid.emit_template(wide)
write_chunk(wide, "1121121", {0: rng.uniform(0.0, 400.0, 5_000)})

# Default fit="raise": the 400 m span overflows the 64 m window.
try:
    next(read_tensors(wide, FIELD, bottom=0.0, top=1.0))
except ValueError as exc:
    print("fit='raise' ->", exc)

In [ ]:
# degrade_resolution: coarsen the bins until the full span fits in 128 bins.
t, _ = next(read_tensors(wide, FIELD, bottom=0.0, top=1.0, fit="degrade_resolution"))
print("fit='degrade_resolution' -> tensor", t.shape, "dtype", t.dtype)

# collapse_bins on a *narrow* chunk: shrink n_bins to the smallest pow2 that fits.
narrow = MemoryStore()
grid.emit_template(narrow)
write_chunk(narrow, "1121121", {0: rng.uniform(100.0, 110.0, 4_000)})
t2, _ = next(read_tensors(narrow, FIELD, bottom=0.0, top=1.0, fit="collapse_bins"))
print("fit='collapse_bins' -> tensor", t2.shape, "(n_bins collapsed to fit ~10 m span)")

## 4. `read_raw_values`: lossless recovery

When a cell's digest was built with **no merges** -- every centroid weight is 1,
which happens whenever a cell has at most `delta` samples -- the centroid means
are exactly the original observations. `read_raw_values` yields
`(morton_index, cell_id, values)` for each such cell, with `values` the recovered
sample vector (sorted ascending, as the digest stores centroids by mean).

If any cell carries a merged centroid (weight > 1), exact recovery is impossible
and the reader raises -- the same `raise`-by-default contract as `read_tensors`.

In [ ]:
from zagg.readers import read_raw_values

lossless = MemoryStore()
grid.emit_template(lossless)
samples = np.array([3.0, 1.0, 2.0, 5.0, 4.0])  # 5 values, well under delta -> no merges
write_chunk(lossless, "1121121", {7: samples}, delta=512)

for morton, cell_id, values in read_raw_values(lossless, FIELD):
    print(f"chunk {morton}, cell {cell_id}: recovered {values}")
    print("matches sorted input:", np.allclose(values, np.sort(samples)))

In [ ]:
# A cell with many samples at a small delta forces merges -> not recoverable.
merged = MemoryStore()
grid.emit_template(merged)
write_chunk(merged, "1121121", {0: rng.standard_normal(5_000)}, delta=64)
try:
    list(read_raw_values(merged, FIELD))
except ValueError as exc:
    print("merged digest ->", exc)

## Summary

- `read_tensors(store, field, ...)` -> dense `(64, 64, n_bins)` tensors per chunk,
  with `bottom`/`top` tail-trim, a `fit` policy for wide chunks, and a `dtype`
  flag (`uint16`/`uint32`/`float32`).
- `read_raw_values(store, field)` -> exact samples per cell when the digest is
  merge-free, raising otherwise.
- `read_cell(store, field, cell)` -> one cell's digest by global position.

All read the ragged (variable-length-bytes) t-digest field zagg writes (issue
#209) and recover the chunk morton id from the sibling `morton` coordinate.
Point them at a real on-disk or S3 Zarr product and the same calls apply -- only
the `store` argument changes.

> **Note:** this notebook uses a synthetic in-memory store so it stays
> Binder-runnable with no credentials. A companion check against a non-synthetic
> (real-product) t-digest store is tracked as a follow-up on issue #79.